# Fase 3: Creación de la especie con Ollama (SageMaker)

Este notebook corre **dentro del notebook de SageMaker**, donde Ollama ya está instalado con Docker (`01_instalar_ollama.sh`).

Qué hace:
1. Comprueba que Ollama responde y lista los modelos descargados.
2. Prueba que cada modelo recibe una instrucción y devuelve una respuesta.
3. Pide la **identidad** del dinosaurio seleccionado (apariencia, hábitat, alimentación, comportamiento y característica distintiva) a los 3 modelos y los compara.
4. Registra el nombre y la versión del modelo elegido.
5. Guarda la identidad en `ollama_setup/identidad_dinosaurio.json` (entrada de las fases siguientes).

**Uso:** menú *Run → Run All Cells*. La comparación tarda varios minutos porque el servidor no tiene GPU.

In [ ]:
import sys, os, json, time, zipfile
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

def find_repo():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'name_generator').exists() and (q / 'ollama_setup').exists():
            return q
    return Path('/home/ec2-user/SageMaker/Dinosaurio')

REPO = find_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'ollama_setup'))
import ollama_client as oc

oc.OLLAMA_URL = 'http://localhost:11434'     # Ollama corre en Docker en esta misma instancia

sel_path = REPO / 'name_generator/artifacts/nombre_seleccionado.json'
if not sel_path.exists():
    raise FileNotFoundError('Falta nombre_seleccionado.json. En la terminal ejecute: cd /home/ec2-user/SageMaker/Dinosaurio && git pull')
sel = json.loads(sel_path.read_text(encoding='utf-8'))
NOMBRE = sel['name']
print('Repositorio:', REPO)
print('Dinosaurio seleccionado en la Fase 1:', NOMBRE, '(modelo de caracteres:', sel.get('model'), ')')

## 1. Verificar Ollama y los modelos descargados

In [ ]:
print('Version de Ollama:', oc.version())
tags = oc.list_models()
modelos = [m['name'] for m in tags]
tabla_modelos = pd.DataFrame([{'modelo': m['name'], 'tamano_GB': round(m['size'] / 1e9, 2),
                               'digest': m['digest'][:12]} for m in tags])
display(tabla_modelos)
assert modelos, 'No hay modelos descargados. Ejecute ollama_setup/01_instalar_ollama.sh'

## 2. Prueba: el modelo recibe una instrucción y devuelve una respuesta

In [ ]:
PRUEBA = 'Responde en una sola frase en español: ¿qué es un dinosaurio?'
for m in modelos:
    r = oc.generate(m, PRUEBA, num_predict=60, temperature=0.3)
    print(f"[{m}] ({r['wall_s']:.0f} s) {r['response'].strip()}\n")

## 3. Identidad del dinosaurio: comparación de los tres modelos

Se envía **el mismo prompt** a cada modelo. El prompt incluye ejemplos breves de sufijos de nombres reales (`-saurus`, `-raptor`, `-odon`, `-venator`) y pide un JSON con cinco campos coherentes entre sí. Si el JSON llega incompleto se reintenta (máximo 3 veces).

In [ ]:
print('Prompt enviado a los modelos:\n')
print(oc.build_identity_prompt(NOMBRE))

In [ ]:
resultados, filas = {}, []
for m in modelos:
    print(f'Generando identidad con {m} ...')
    res = oc.generar_identidad(NOMBRE, m, temperature=0.7, seed=42)
    resultados[m] = res
    filas.append({'modelo': m, 'json_valido': res['ok'], 'intentos': res['intentos'],
                  'campos_ok': len(res['campos']), 'palabras': len(res['descripcion'].split()),
                  'tokens_por_s': round(res['metrics'].get('tokens_s', 0), 1),
                  'tiempo_total_s': round(res['total_s'])})
comparacion = pd.DataFrame(filas)
display(comparacion)
for m, res in resultados.items():
    print('=' * 72); print(m, '| JSON válido:', res['ok']); print('-' * 72); print(res['descripcion']); print()

### 3.1 Elegir el modelo

Lea las tres descripciones y elija con estos criterios: **coherencia** entre los rasgos (la dieta encaja con la dentadura y el tamaño, el hábitat con la apariencia), **español natural**, que use el nombre/sufijo como inspiración, y **velocidad** (`tokens_por_s`). Escriba su elección en `MODELO_ELEGIDO` (por defecto `qwen2.5:7b`) y ejecute esta celda y las siguientes.

In [ ]:
MODELO_ELEGIDO = 'qwen2.5:7b'   # <-- cambie por llama3.1:8b o mistral:7b si prefiere otra descripción
assert MODELO_ELEGIDO in resultados, f'Elija uno de: {list(resultados)}'

info = oc.info_modelo(MODELO_ELEGIDO)
info.update({'ollama_version': oc.version(), 'instancia': 'ml.m5.xlarge (CPU)',
             'fecha_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')})
Path('docs').mkdir(exist_ok=True)
Path('docs/modelo_ollama.json').write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
res = resultados[MODELO_ELEGIDO]
if not res['ok']:
    print('La respuesta del modelo elegido no fue válida; se vuelve a generar...')
    res = oc.generar_identidad(NOMBRE, MODELO_ELEGIDO, temperature=0.7, seed=7, max_intentos=5)
assert res['ok'], 'No se obtuvo una identidad válida; pruebe con otro modelo.'

identidad = {
    'nombre': NOMBRE,
    'modelo_ollama': MODELO_ELEGIDO,
    'digest_modelo': info['digest'],
    **res['campos'],
    'descripcion': res['descripcion'],
    'generacion': {'temperature': 0.7, 'intentos': res['intentos']},
}
Path('ollama_setup/identidad_dinosaurio.json').write_text(json.dumps(identidad, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Identidad guardada: ollama_setup/identidad_dinosaurio.json\n')
print(NOMBRE); print(identidad['descripcion'])

## 4. Evidencias para el repositorio

In [ ]:
def df_to_md(df):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for _, row in df.iterrows():
        lines.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
    return '\n'.join(lines)

Path('docs/evidencias').mkdir(parents=True, exist_ok=True)
Path('docs/evidencias/ollama_comparacion.json').write_text(json.dumps(
    {m: {'ok': r['ok'], 'intentos': r['intentos'], 'metrics': r['metrics'], 'campos': r['campos'],
         'descripcion': r['descripcion']} for m, r in resultados.items()},
    indent=2, ensure_ascii=False), encoding='utf-8')

informe = f"""# Ollama: comparación de modelos e identidad del dinosaurio

Instancia: SageMaker ml.m5.xlarge (CPU, 16 GB RAM). Ollama {oc.version()} en Docker.
Dinosaurio: **{NOMBRE}**. Mismo prompt, temperatura 0.7, semilla 42.

## Modelos descargados
{df_to_md(tabla_modelos)}

## Resultados
{df_to_md(comparacion)}

## Modelo elegido
**{MODELO_ELEGIDO}** (digest `{info['digest'][:12]}`, {info.get('parameter_size')}, cuantización {info.get('quantization_level')}).

### Justificación (completar con sus palabras)
- Coherencia entre los rasgos de las tres descripciones:
- Calidad del español:
- Velocidad en CPU (tokens por segundo):

## Identidad generada
{identidad['descripcion']}
"""
Path('docs/comparacion_modelos_ollama.md').write_text(informe, encoding='utf-8')

archivos = ['docs/modelo_ollama.json', 'docs/comparacion_modelos_ollama.md', 'docs/evidencias/ollama_instalacion.txt',
            'docs/evidencias/ollama_comparacion.json', 'ollama_setup/identidad_dinosaurio.json']
salida = REPO.parent / 'fase3_resultados.zip'        # fuera del repo, en /home/ec2-user/SageMaker
with zipfile.ZipFile(salida, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        if Path(a).exists():
            z.write(a)
        else:
            print('Falta (se omite):', a)
print('Creado:', salida)

### Siguiente
1. En el explorador de archivos de la izquierda (carpeta `SageMaker`), clic derecho sobre `fase3_resultados.zip` → **Download**.
2. Descomprímalo en la raíz del repo en su computador y complete la justificación en `docs/comparacion_modelos_ollama.md`.
3. `git add -A && git commit -m "fase-3: ollama + identidad" && git pull origin main --rebase && git push`
4. Exponer Ollama con ngrok: `bash ollama_setup/03_exponer_ngrok.sh` (en la terminal de SageMaker).